# Stage 1: NDT-Transformer on KITTI sequence 00

Runtime -> Change runtime type -> **T4 GPU** (or better).

First session: run every cell from top to bottom.
After a disconnect: run cells 1-3, then **3b** (restore the NDT cells from Drive), then 5 and 7. Training resumes from the last checkpoint.

## 1. Settings

In [ ]:
# Where the code comes from: your GitHub repo URL, or leave empty to use a copy on Drive.
REPO_URL = ""                                   # e.g. "https://github.com/<you>/lidar-place-recognition.git"
DRIVE = "/content/drive/MyDrive/lpr"            # everything that must survive goes here
CODE_ON_DRIVE = f"{DRIVE}/code/lidar-place-recognition"

KITTI = "/content/kitti"                        # raw scans (local disk, temporary)
NDT = "/content/ndt"                            # NDT cells (local copy)
NDT_TAR = f"{DRIVE}/ndt_kitti00_patchwork.tar"  # NDT cells (permanent copy on Drive)
RUN = f"{DRIVE}/runs/ndt_kitti00"               # checkpoints and metrics

# Mixed precision saves GPU memory and time (a documented deviation from the paper).
TRAIN_FLAGS = "--amp"

## 2. Drive, GPU, code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

if REPO_URL:
    # For a private repo, store a token in Colab Secrets as GITHUB_TOKEN.
    url = REPO_URL
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
        url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    if not os.path.exists("/content/lpr"):
        subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
else:
    !rm -rf /content/lpr && cp -r "$CODE_ON_DRIVE" /content/lpr
%cd /content/lpr
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt

## 3. Download sequence 00 (first session only)
About 8 GB, read directly out of the KITTI zip. Poses are always downloaded (4 MB).

In [ ]:
if not os.path.exists(NDT_TAR):
    !python scripts/download_kitti.py --out $KITTI --seqs 00 --threads 16
else:
    !python scripts/download_kitti.py --out $KITTI --seqs 00 --poses_only
    print("NDT cells already on Drive, scans not needed. Run cell 3b.")

### 3b. Restore NDT cells from Drive (later sessions)

In [ ]:
if os.path.exists(NDT_TAR) and not os.path.exists(f"{NDT}/00/000000.bin"):
    !mkdir -p $NDT && tar -xf "$NDT_TAR" -C $NDT
!ls $NDT/00 | wc -l

## 4. G on one scan, then on all of sequence 00 (first session only)
Look at the picture before running everything: the red ellipses should sit on walls, cars, poles and trees, not on the road.

In [ ]:
from IPython.display import Image, display
!python scripts/visualize_ndt.py --kitti_root $KITTI --seq 00 --idx 0 --out /content/cells_000000.png
display(Image("/content/cells_000000.png"))

In [ ]:
if not os.path.exists(NDT_TAR):
    !python scripts/extract_ndt.py --kitti_root $KITTI --seq 00 --out_dir $NDT --ground patchwork
    !tar -cf "$NDT_TAR" -C $NDT .
    !cp $NDT/00_stats.csv $DRIVE/
import pandas as pd
st = pd.read_csv(f"{NDT}/00_stats.csv")
print(st[["n_raw", "n_points", "voxel", "cells_before_removal", "removed", "padded", "seconds"]].describe().round(2))

## 5. Training tuples (frames 0-2999, positive < 5 m, negative > 7 m)

In [ ]:
!python scripts/make_kitti_tuples.py --kitti_root $KITTI --seq 00 --ndt_dir $NDT --out $NDT/train_00.pickle

## 6. Timing and memory test (30 iterations)

If the full batch (2 x 22 clouds) does not fit in GPU memory, the script switches by itself to one tuple per forward pass (`--split_forward`) and prints a message. Note the seconds per iteration and the peak memory.
The epoch time is roughly (seconds per iteration) x 1500.

In [ ]:
!rm -rf /content/smoke
!python scripts/train_ndt_kitti.py --ndt_dir $NDT --kitti_root $KITTI --train_pickle $NDT/train_00.pickle \
    --log_dir /content/smoke --epochs 1 --max_iters 30 --eval_every 0 $TRAIN_FLAGS 2>&1 | grep -v Warning

## 7. Full training (20 epochs, resumes automatically)
Checkpoints every 300 iterations and at the end of every epoch go to Drive.

In [ ]:
!python scripts/train_ndt_kitti.py --ndt_dir $NDT --kitti_root $KITTI --train_pickle $NDT/train_00.pickle \
    --log_dir "$RUN" $TRAIN_FLAGS 2>&1 | grep -v Warning

## 8. Results

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
m = pd.read_csv(f"{RUN}/metrics.csv")
display(m)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(m.epoch, m.train_loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(m.epoch, m["R@1"], marker="o", label="R@1"); ax[1].plot(m.epoch, m["R@1%"], marker="o", label="R@1%")
ax[1].set_title("KITTI 00 recall (%)"); ax[1].set_xlabel("epoch"); ax[1].legend()
print("total training time: %.1f hours" % (m.epoch_minutes.sum() / 60))

In [ ]:
last = sorted(f for f in os.listdir(RUN) if f.startswith("epoch_"))[-1]
!python scripts/eval_ndt_kitti.py --ckpt "$RUN/$last" --ndt_dir $NDT --kitti_root $KITTI --seq 00 --out "$RUN/eval_00_$last.json" 2>&1 | grep -v Warning